# 15｜電腦視覺_遷移學習與偵測分割

用 IoU 與 NMS 小例子連接偵測框、信心分數與後處理。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
boxes = np.array([[0.10,0.10,0.55,0.55], [0.16,0.14,0.58,0.57], [0.62,0.58,0.92,0.90]])
scores = np.array([0.92, 0.81, 0.74])
def iou(a, b):
    lo = np.maximum(a[:2], b[:2]); hi = np.minimum(a[2:], b[2:])
    inter = np.prod(np.maximum(hi - lo, 0))
    area_a = np.prod(a[2:] - a[:2]); area_b = np.prod(b[2:] - b[:2])
    return inter / (area_a + area_b - inter)
keep = []
for idx in np.argsort(scores)[::-1]:
    if all(iou(boxes[idx], boxes[j]) < 0.5 for j in keep): keep.append(int(idx))
table = pd.DataFrame(boxes, columns=['x1','y1','x2','y2']).assign(score=scores, kept=[i in keep for i in range(3)])
print('IoU(box 0, box 1)=', round(iou(boxes[0], boxes[1]), 3))
table

## 執行結果圖

In [ ]:
from matplotlib.patches import Rectangle
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, ids, title in [(axes[0], range(3), 'before NMS'), (axes[1], keep, 'after NMS')]:
    for i in ids:
        x1,y1,x2,y2 = boxes[i]
        ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, lw=3, label=f'{scores[i]:.2f}'))
    ax.set(xlim=(0,1), ylim=(1,0), title=title); ax.legend()
savefig('15_detection_demo')
report('vision_detection', {'keep': keep, 'iou_0_1': iou(boxes[0], boxes[1])})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。